In [ ]:
!nvidia-smi

Mon Sep 29 09:12:59 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A40                     Off |   00000000:65:00.0 Off |                    0 |
|  0%   29C    P8             24W /  300W |       1MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
# load required libraries
import sys
sys.path.append("../utils")
from pairing_utils import iou_batch
from json_parser import parse_json_annotations, show_sample, TestDataSet, OPTICAL_CHARACTERISTICS
from cv_utils import show_detections
from precision_recall_eval import evaluate_yolo_pr, evaluate_mask_rcnn_pr, calculate_confusion_matrix, p_r_based_on_c_m
import numpy as np
import pandas as pd
import os
import cv2
from typing import List, Dict, Union, Tuple, Final
from yolo_model import run_yolo_v5, detector

In [13]:
!readlink -f ../../datasets/

/global/home/ashish.sinha/cellanome/datasets


In [14]:
# NOTE: We are not using any objset size filtering below, it helps with 10x datasets to use 6um for the smallest object
# size, 
# for 4x, it significantly drops PRECISION as many small objects are still being detected by the model

reverse_label_map = {'cell': 0, 'Cell': 0, 'dying/dead cells': 0, 'dead-cell': 0,
                     'Bead': 1, 'bead': 1, 
                     'spheroid': 2,
                    }

dataset = TestDataSet(dataset_paths = [# '/home/cellareye/Cellanome/Data/old_microscope_data_and_old_analysis_data_sets_1_2_test_set',
                                       # '/home/cellareye/Cellanome/Data/20240228_jurkat_10x_caged',
                                       # '/home/cellareye/Cellanome/Data/20240228_jurkat_10x_uncaged',
                                       # '/home/cellareye/Cellanome/Data/20240228_k562_10x_caged',
                                       # '/home/cellareye/Cellanome/Data/20240228_k562_10x_uncaged',
                                       # '/home/cellareye/Cellanome/Data/20240228_nk92_10x_caged',
                                       # '/home/cellareye/Cellanome/Data/20240228_nk92_10x_uncaged',
                                       # '/home/cellareye/Cellanome/Data/20240425_nk92_10x_caged',
                                       # '/home/cellareye/Cellanome/Data/20240425_nk92_10x_uncaged',
                                       # '/home/cellareye/Cellanome/Data/20240228_hela-suspension_10x_caged',
                                       # '/home/cellareye/Cellanome/Data/20240228_hela-suspension_10x_uncaged', 
                                       # '/home/cellareye/Cellanome/Data/20240314_imr90-suspension_10x_caged', 
                                       # '/home/cellareye/Cellanome/Data/20240314_imr90-suspension_10x_uncaged',
                                       # '/home/cellareye/Cellanome/Data/20240307_pbmc-beads_10x_uncaged',
                                       # '/home/cellareye/Cellanome/Data/20240305_pbmc-nobeads_10x_caged',
                                       # '/home/cellareye/Cellanome/Data/20240305_pbmc-nobeads_10x_uncaged',
                                       # '/home/cellareye/Cellanome/Data/20240306_mousepbmc-beads_10x_caged',
                                       # '/home/cellareye/Cellanome/Data/20240306_mousepbmc-beads_10x_uncaged',
                                       # '/home/cellareye/Cellanome/Data/20240306_mousepbmc-nobeads_10x_caged',
                                       # '/home/cellareye/Cellanome/Data/20240306_mousepbmc-nobeads_10x_uncaged',
                                       # '/home/cellareye/Cellanome/Data/20240422_neuron-adhered_10x_uncaged',
                                        # uncomment the below lines from here 
                                       # '/home/cellareye/Cellanome/Data/20240221_jurkat_4x_caged',  
                                       # '/home/cellareye/Cellanome/Data/20240221_jurkat_4x_uncaged',  
                                       # '/home/cellareye/Cellanome/Data/20240226_hela-suspension_4x_uncaged',  
                                       # '/home/cellareye/Cellanome/Data/20240305_pbmc-nobeads_4x_uncaged',  
                                       # '/home/cellareye/Cellanome/Data/20240320_mpbmc-beads_4x_uncaged',  
                                       # '/home/cellareye/Cellanome/Data/20240821_neurosphere_4x_uncaged'
                                        # ASHISH
                                        '/global/home/ashish.sinha/cellanome/datasets/20240221_jurkat_4x_caged/',
                                      ], 
                      scale_factor_dict = {}, 
                      max_larger_side = 5000,
                      max_smaller_side = 5000,
                      min_object_diameter = 6.0, # NOTE: this should be set to 0 for 4x, this is done automatically below
                      class_names_to_ids_map = reverse_label_map,  
                      labels_of_interest = list(reverse_label_map.keys()))

FileNotFoundError: [Errno 2] No such file or directory: '/global/home/ashish.sinha/cellanome/datasets/20240221_jurkat_4x_caged/test_annotations'

In [ ]:
len(dataset)

In [ ]:
if detector.get_metadata()['magnification'].lower() == '4x':
    is_4x=True
    print("[INFO]: The model is specific to 4x images")
    print("[INFO]: Setting the min_object_diameter filter of the Dataset class to 0.0 as 4x datasets contain many small annotated objects!")
    dataset.min_object_diameter = 0.0
else:
    is_4x=False
    print("[INFO]: The model is specific to 10x images")

found_4x: bool = False
found_10x: bool = False
for dataset_path in dataset.dataset_paths:
    if '_4x_' in os.path.basename(dataset_path).lower():
        found_4x = True
    if '_10x_' in os.path.basename(dataset_path).lower():
        found_10x = True

if found_4x and found_10x:
    print('[ERROR]: The test datasets are a mix of 4x and 10x images! This should never happen')
elif found_4x and not is_4x:
    print('[WARNING]: The test datasets are 4x but the configured mode is for 10x images! Make sure the correct model is being used')
elif found_10x and is_4x:
    print('[WARNING]: The test datasets are 10x but the configured mode is for 4x images! Make sure the correct model is being used')

if found_4x and not found_10x:
    print("[INFO]: The datasets are all 4x images")
elif found_10x and not found_4x:
    print("[INFO]: The datasets are all 10x images")
elif not found_10x and not found_4x:
    print('[WARNING]: The test dataset names do not indicate if they are 4x or 10x! Manually make sure the correct model is used')

In [ ]:
predictions: List = []
run_times: List = []
cell_areas: List = []

for idx in range(len(dataset)):
    annots = dataset[idx]

    cell_areas += [np.sum(mask) for i, mask in enumerate(annots['masks']) if 
                   annots['annotations'].loc[i]['label'] == reverse_label_map['cell']]
    
    boxes, labels, scores, run_time = run_yolo_v5(annots['image'], is_4x=is_4x)
    
    run_times.append(run_time)
    
    # no need to filter for labels_of_interest here as we are doing it during precision and 
    # recall evaluation, similary, no need to only return the classes of interest from the
    # dataset
    
    predictions.append({'boxes': boxes, 'labels': labels, 'scores': scores})
print(f"YOLOv5 took {np.mean(run_times[1:]) * 1000}ms on average per image")

In [ ]:
# None means all, 
# labels_of_interest = None
model_label_map: Dict[int, str] = detector._label_map
precision = {}
recall = {}
for class_id, class_name in model_label_map.items():
    # use the same function for calculating precision and recall of the Mask RCNN
    # detections if you do not want to use mask IoU (use box IoU)
    # precision, recall = evaluate_yolo_pr(predictions, dataset, class_ids_of_interest, 0.5)
    precision[class_name], recall[class_name] = evaluate_yolo_pr(predictions, dataset, [class_id], 0.5)
    print(f"Completed PR calculation for '{class_name}' with ID: {class_id}")

for class_name in precision:
    print(f"YOLOv5 Precision: {precision[class_name]}, Recall: {recall[class_name]} at IoU 0.5 for labels: {class_name}")
    if precision[class_name] + recall[class_name] > 0:
        print(f"YOLOv5 F-1 Score: {2 * precision[class_name] * recall[class_name] / (precision[class_name] + recall[class_name])}", 
              f"at IoU 0.5 for labels: {class_name}")

In [ ]:
c_m_df, p_r_df, p_r_with_break_down_df = p_r_based_on_c_m(predictions=predictions, 
                                                          dataset=dataset, 
                                                          class_ids_to_classnames_map=model_label_map,
                                                          model_label_map=model_label_map, 
                                                          annotation_filter=None,
                                                          use_masks=False)

In [ ]:
p_r_df

In [ ]:
c_m_df

In [ ]:
from PIL import Image
idx = 5
annots = dataset[idx]
img = show_sample(sample=annots, class_id_to_name_mapping=yolo_model.detector._label_map)
Image.fromarray(img)

In [ ]:
display(Image.fromarray(annots['image']))

In [ ]:
img = show_detections(input_image=annots['image'], predictions=predictions[idx], label_map=yolo_model.detector._label_map)
Image.fromarray(img)

In [ ]:
from pairing_utils import pair_gts_dets_bbox
def evaluate_pr_per_sample(predictions, annotations, class_ids_of_interest=None, min_iou=0.5):
    
    num_true_positives: int = 0
    num_false_positives: int = 0
    num_false_negatives: int = 0
    
    boxes = predictions['boxes']
    labels = predictions['labels']
        
    if class_ids_of_interest is None:
        # use the union of all the class IDs from the detections and the annotations
        class_ids_to_filter = list(annotations['annotations']['label'].unique())
        class_ids_to_filter += list(np.unique(labels))
        # remove duplicates
        class_ids_to_filter = list(set(class_ids_to_filter))
    else:
        class_ids_to_filter = class_ids_of_interest
            
    for class_id in class_ids_to_filter:
        # filter the detections and ground truths for the given label
        gt_boxes = annotations['annotations'].loc[annotations['annotations']['label'] == class_id, ['xtl', 'ytl', 'xbr', 'ybr']].values.astype(int)
        det_boxes = boxes[labels == class_id, :]
        # pair
        paired_idx, unpaired_gts, unpaired_dets = pair_gts_dets_bbox(gt_boxes, det_boxes, min_iou)
        
        num_true_positives += len(paired_idx)
        num_false_positives += len(unpaired_dets)
        num_false_negatives += len(unpaired_gts)
        
        
    return num_true_positives / (num_true_positives + num_false_positives + 1e-30), num_true_positives / (num_true_positives + num_false_negatives + 1e-30)

In [ ]:
evaluate_pr_per_sample(predictions=predictions[idx], annotations=annots, class_ids_of_interest=[0], min_iou=0.5)

In [ ]:
"""
###  All suspension cells

* Existing model
YOLOv5 Precision: 0.9168021597079485, Recall: 0.9040445291466255 at IoU 0.5 for labels: cell
YOLOv5 F-1 Score: 0.910378651719621 at IoU 0.5 for labels: cell

YOLOv5 Precision: 0.9578377963630783, Recall: 0.9620453144266338 at IoU 0.5 for labels: bead
YOLOv5 F-1 Score: 0.9599369449036872 at IoU 0.5 for labels: bead

* 20)
YOLOv5 Precision: 0.9335877343962315, Recall: 0.9271312424797868 at IoU 0.5 for labels: cell
YOLOv5 F-1 Score: 0.9303482867766124 at IoU 0.5 for labels: cell

YOLOv5 Precision: 0.9828837839581854, Recall: 0.9549985892974702 at IoU 0.5 for labels: bead
YOLOv5 F-1 Score: 0.9687405593627272 at IoU 0.5 for labels: bead

* 26)
YOLOv5 Precision: 0.9503584891714095, Recall: 0.9354510134889339 at IoU 0.5 for labels: cell
YOLOv5 F-1 Score: 0.9428458289334741 at IoU 0.5 for labels: cell

YOLOv5 Precision: 0.9850093033597416, Recall: 0.961185780680819 at IoU 0.5 for labels: bead
YOLOv5 F-1 Score: 0.9729517292398742 at IoU 0.5 for labels: bead

* 27)
YOLOv5 Precision: 0.9493542669030134, Recall: 0.929967533651917 at IoU 0.5 for labels: cell
YOLOv5 F-1 Score: 0.9395609053149105 at IoU 0.5 for labels: cell

YOLOv5 Precision: 0.9825948995181573, Recall: 0.9581265828587144 at IoU 0.5 for labels: bead
YOLOv5 F-1 Score: 0.97020649481007 at IoU 0.5 for labels: bead

###  All cell types

* 27)
YOLOv5 Precision: 0.9437272449841381, Recall: 0.9222841051748174 at IoU 0.5 for labels: cell
YOLOv5 F-1 Score: 0.9328824689036839 at IoU 0.5 for labels: cell

YOLOv5 Precision: 0.9825718046349833, Recall: 0.9581265828587144 at IoU 0.5 for labels: bead
YOLOv5 F-1 Score: 0.9701952365756722 at IoU 0.5 for labels: bead

YOLOv5 Precision: 0.8810735770476631, Recall: 0.9611307420494699 at IoU 0.5 for labels: soma
YOLOv5 F-1 Score: 0.9193626267503622 at IoU 0.5 for labels: soma

### Jurkat sets only

* Existing model
YOLOv5 Precision: 0.9554953560371517, Recall: 0.9247191011235955 at IoU 0.5 for labels: cell
YOLOv5 F-1 Score: 0.9398553483060526 at IoU 0.5 for labels: cell

* 20)
YOLOv5 Precision: 0.9684130887081521, Recall: 0.9588014981273408 at IoU 0.5 for labels: cell
YOLOv5 F-1 Score: 0.9635833254916722 at IoU 0.5 for labels: cell

* 26)
YOLOv5 Precision: 0.9715577321529478, Recall: 0.9635718288810013 at IoU 0.5 for labels: cell
YOLOv5 F-1 Score: 0.9675483023822923 at IoU 0.5 for labels: cell

* 27)
YOLOv5 Precision: 0.9705827243769908, Recall: 0.9691300280636108 at IoU 0.5 for labels: cell
YOLOv5 F-1 Score: 0.9698558322411533 at IoU 0.5 for labels: cell

### K562 sets only

* Existing model
YOLOv5 Precision: 0.9299595141700405, Recall: 0.9678370786516854 at IoU 0.5 for labels: cell
YOLOv5 F-1 Score: 0.9485203028217482 at IoU 0.5 for labels: cell

* 20)
YOLOv5 Precision: 0.9577953630431765, Recall: 0.9689606741573034 at IoU 0.5 for labels: cell
YOLOv5 F-1 Score: 0.9633456678070237 at IoU 0.5 for labels: cell

* 26)
YOLOv5 Precision: 0.9632055749128919, Recall: 0.9691487869863974 at IoU 0.5 for labels: cell
YOLOv5 F-1 Score: 0.9661680413812387 at IoU 0.5 for labels: cell

* 27)
YOLOv5 Precision: 0.9617471136458479, Recall: 0.9698414924954412 at IoU 0.5 for labels: cell
YOLOv5 F-1 Score: 0.9657773432043582 at IoU 0.5 for labels: cell

### NK92 sets only

* Existing model
YOLOv5 Precision: 0.8486265341905318, Recall: 0.8546203649205415 at IoU 0.5 for labels: cell
YOLOv5 F-1 Score: 0.8516129032258065 at IoU 0.5 for labels: cell

* 20)
YOLOv5 Precision: 0.9199324731701435, Recall: 0.898057680988817 at IoU 0.5 for labels: cell
YOLOv5 F-1 Score: 0.9088634739099357 at IoU 0.5 for labels: cell

* 26)
YOLOv5 Precision: 0.9373193890188524, Recall: 0.9008133306883555 at IoU 0.5 for labels: cell
YOLOv5 F-1 Score: 0.9187038473210372 at IoU 0.5 for labels: cell

* 27) 
YOLOv5 Precision: 0.9331514324693042, Recall: 0.9050611974859412 at IoU 0.5 for labels: cell
YOLOv5 F-1 Score: 0.9188916876574308 at IoU 0.5 for labels: cell

### HeLa suspension sets only

* Existing model
YOLOv5 Precision: 0.9547609893630655, Recall: 0.9665282823040996 at IoU 0.5 for labels: cell
YOLOv5 F-1 Score: 0.9606086003481401 at IoU 0.5 for labels: cell

* 20)
YOLOv5 Precision: 0.9672236503856041, Recall: 0.9762584327970939 at IoU 0.5 for labels: cell
YOLOv5 F-1 Score: 0.971720041322314 at IoU 0.5 for labels: cell

* 26)
YOLOv5 Precision: 0.9788523533204384, Recall: 0.97998967208882 at IoU 0.5 for labels: cell
YOLOv5 F-1 Score: 0.9794206825366105 at IoU 0.5 for labels: cell

* 27)
YOLOv5 Precision: 0.9762241357152037, Recall: 0.9806351665375678 at IoU 0.5 for labels: cell
YOLOv5 F-1 Score: 0.978424679590391 at IoU 0.5 for labels: cell

### IMR-90 suspension sets only

* Existing model
YOLOv5 Precision: 0.7576301615798923, Recall: 0.9009874566319722 at IoU 0.5 for labels: cell
YOLOv5 F-1 Score: 0.823113495062782 at IoU 0.5 for labels: cell

* 20)
YOLOv5 Precision: 0.8724058416602614, Recall: 0.9087269815852682 at IoU 0.5 for labels: cell
YOLOv5 F-1 Score: 0.8901960784313726 at IoU 0.5 for labels: cell

* 26)
YOLOv5 Precision: 0.8775298391281785, Recall: 0.9023479188900747 at IoU 0.5 for labels: cell
YOLOv5 F-1 Score: 0.8897658510918179 at IoU 0.5 for labels: cell

* 27)
YOLOv5 Precision: 0.8667519181585678, Recall: 0.9042155816435432 at IoU 0.5 for labels: cell
YOLOv5 F-1 Score: 0.8850874902063202 at IoU 0.5 for labels: cell

### PBMC cells (mouse/human with some sets with beads)

* Existing model
YOLOv5 Precision: 0.8912477530527491, Recall: 0.7779310717957042 at IoU 0.5 for labels: cell
YOLOv5 F-1 Score: 0.8307430090131732 at IoU 0.5 for labels: cell

YOLOv5 Precision: 0.8919077175313268, Recall: 0.9730834752981261 at IoU 0.5 for labels: bead
YOLOv5 F-1 Score: 0.9307289651098845 at IoU 0.5 for labels: bead

* 20)
YOLOv5 Precision: 0.9211205712716287, Recall: 0.9072661364497105 at IoU 0.5 for labels: cell
YOLOv5 F-1 Score: 0.9141408634976014 at IoU 0.5 for labels: cell

YOLOv5 Precision: 0.9823497198702557, Recall: 0.9931856899488927 at IoU 0.5 for labels: bead
YOLOv5 F-1 Score: 0.9877379868273364 at IoU 0.5 for labels: bead

* 26)
YOLOv5 Precision: 0.9532924664014204, Recall: 0.9343656318276511 at IoU 0.5 for labels: cell
YOLOv5 F-1 Score: 0.9437341630047791 at IoU 0.5 for labels: cell

YOLOv5 Precision: 0.9867340893703453, Recall: 0.9930795110940761 at IoU 0.5 for labels: bead
YOLOv5 F-1 Score: 0.9898966315029823 at IoU 0.5 for labels: bead

* 27)
YOLOv5 Precision: 0.9518323657474601, Recall: 0.9365046983973174 at IoU 0.5 for labels: cell
YOLOv5 F-1 Score: 0.9441063245907751 at IoU 0.5 for labels: cell

YOLOv5 Precision: 0.9857306177753313, Recall: 0.991798913622324 at IoU 0.5 for labels: bead
YOLOv5 F-1 Score: 0.9887554550377472 at IoU 0.5 for labels: bead

### Neurons

* 27)
YOLOv5 Precision: 0.847870182555781, Recall: 0.8226033174023053 at IoU 0.5 for labels: cell
YOLOv5 F-1 Score: 0.8350456621004566 at IoU 0.5 for labels: cell

YOLOv5 Precision: 0.8692515779981965, Recall: 0.9732458354366481 at IoU 0.5 for labels: soma
YOLOv5 F-1 Score: 0.9183138842581566 at IoU 0.5 for labels: soma

"""

In [ ]:
"""
### 4x images

### All suspension cells

* 26) (model trained on 10x images but applied to 4x images)
YOLOv5 Precision: 0.9214418491877475, Recall: 0.6830013479107384 at IoU 0.5 for labels: cell
YOLOv5 F-1 Score: 0.7845039651820888 at IoU 0.5 for labels: cell

YOLOv5 Precision: 0.8586356120043627, Recall: 0.8204464465810529 at IoU 0.5 for labels: bead
YOLOv5 F-1 Score: 0.8391067407038113 at IoU 0.5 for labels: bead


* 5) (model trained on 4x images)
YOLOv5 Precision: 0.957281844035324, Recall: 0.9578553242474165 at IoU 0.5 for labels: cell
YOLOv5 F-1 Score: 0.9575684982781854 at IoU 0.5 for labels: cell

YOLOv5 Precision: 0.983192151873305, Recall: 0.9871915551670812 at IoU 0.5 for labels: bead
YOLOv5 F-1 Score: 0.9851877946085585 at IoU 0.5 for labels: bead

* 6) (model trained on 4x images)
YOLOv5 Precision: 0.9632563979909112, Recall: 0.9650741350906096 at IoU 0.5 for labels: cell
YOLOv5 F-1 Score: 0.9641644097975551 at IoU 0.5 for labels: cell

YOLOv5 Precision: 0.9884189325276939, Recall: 0.9898809924023398 at IoU 0.5 for labels: bead
YOLOv5 F-1 Score: 0.9891494221983338 at IoU 0.5 for labels: bead

YOLOv5 Precision: 0.8181818181818182, Recall: 0.9 at IoU 0.5 for labels: spheroid
YOLOv5 F-1 Score: 0.8571428571428572 at IoU 0.5 for labels: spheroid



### Jurkat sets only

* 5)
YOLOv5 Precision: 0.9673303167420815, Recall: 0.9586547085201794 at IoU 0.5 for labels: cell
YOLOv5 F-1 Score: 0.9629729729729729 at IoU 0.5 for labels: cell

* 6) 
YOLOv5 Precision: 0.9698887783705579, Recall: 0.9619730941704036 at IoU 0.5 for labels: cell
YOLOv5 F-1 Score: 0.9659147192579585 at IoU 0.5 for labels: cell

### HeLa sets only

* 5) 
YOLOv5 Precision: 0.9798131967460079, Recall: 0.998158379373849 at IoU 0.5 for labels: cell
YOLOv5 F-1 Score: 0.9889007146115251 at IoU 0.5 for labels: cell

* 6)
YOLOv5 Precision: 0.9824932085722909, Recall: 0.9990791896869244 at IoU 0.5 for labels: cell
YOLOv5 F-1 Score: 0.9907167858773398 at IoU 0.5 for labels: cell

### PBMC cells (mouse/human with some sets with beads)

* 5)
YOLOv5 Precision: 0.9475204874973734, Recall: 0.9504663540074827 at IoU 0.5 for labels: cell
YOLOv5 F-1 Score: 0.9489911346118434 at IoU 0.5 for labels: cell

YOLOv5 Precision: 0.9832250719882141, Recall: 0.9871915551670812 at IoU 0.5 for labels: bead
YOLOv5 F-1 Score: 0.9852043212775952 at IoU 0.5 for labels: bead

* 6)
YOLOv5 Precision: 0.9565217391304348, Recall: 0.9610581229909891 at IoU 0.5 for labels: cell
YOLOv5 F-1 Score: 0.9587845652402482 at IoU 0.5 for labels: cell

YOLOv5 Precision: 0.9884189325276939, Recall: 0.9898809924023398 at IoU 0.5 for labels: bead
YOLOv5 F-1 Score: 0.9891494221983338 at IoU 0.5 for labels: bead

### Neurospheres (spheroid) only

* 6) 
YOLOv5 Precision: 0.8181818181818182, Recall: 0.9 at IoU 0.5 for labels: spheroid
YOLOv5 F-1 Score: 0.8571428571428572 at IoU 0.5 for labels: spheroid

### Cells only datasets and forcing all detections to be cells

* 26) (model trained on 10x images but applied to 4x images)
YOLOv5 Precision: 0.8807957038045534, Recall: 0.766849955549702 at IoU 0.5 for labels: cell
YOLOv5 F-1 Score: 0.8198827733089258 at IoU 0.5 for labels: cell

* 5) (model trained on 4x images)
YOLOv5 Precision: 0.9641669136778316, Recall: 0.9639129432682493 at IoU 0.5 for labels: cell
YOLOv5 F-1 Score: 0.9640399117463035 at IoU 0.5 for labels: cell

"""

## Precision and Recall calculated for different models and over different test sets

This section documents the precision and recall values for all the Mask R-CNN models trained so far. For each model identified by a name, the list of classnames the model is supposed to detect, the datasets the model is trained on, as well as the datasets the model is test on are provided.

For the datasets, the indexes used in the table are defined below:

| Set index | Processed dataset name | Darwin dataset names | Annotated classes | Crop overlaps x, y | Comments |
|:----------|:----------|:----------|:----------|:----------|:----------|
| 1 | `old_microscope_data_176_160` |  `D1` |  'Cell'<br>'Bead' <br> 'dying/dead cells' <br> 'Cluster'| 176, 160| Old ix-81 microscope dataset|
| 2 | `old_analysis_data_set_1_176_160` |  `Normalised_cytokine_PBMC_10312022` <br> `Normalised_cytokine_NK_10312022` <br> `Normalised_surface_Jurkat_10312022` <br> `Normalised_surface_NK_Jurkat_10312022` <br> `Normalised_surface_PBMC_10312022` |  'Cell'<br>'Bead' <br> 'cages'| 176, 160| Mix of ix-81 and BB2|
| 3 | `old_analysis_data_set_2_176_160` | `Normalised_12212022_tregs_beads_cages_BB2` |  'Cell'<br>'Bead' <br> 'cages'| 176, 160| Old BB2|
| 4 | `imr_90_nucleus_cytoplasm_sets_1_2_458_416`| `230607_IMR90_training_dataset_1_cytoplasm` <br> `230607_IMR90_training_dataset_1_nuclei` <br> `230622_IMR90_training_dataset_3_cytoplasm` <br> `230622_IMR90_training_dataset_3_nuclei`| 'nucleus' <br> 'cytoplasm' | 458, 416 | Early versions of multi-channel datasets annotated separately|
| 5 | `imr_90_nucleus_cytoplasm_cage_set_3_458_416`| `230915_IMR90_training_dataset_3_cytoplasm` <br> `230915_IMR90_training_dataset_3_nuclei` <br> `230915_IMR90_training_dataset_3_cages_bf`| 'nucleus' <br> 'cytoplasm' <br> 'cages'| 458, 416 | Early versions of multi-channel datasets annotated separately|
| 6 |`imr_90_cell_nucleus_cytoplasm_cage_sets_4_5_458_416`| `231212_imr90_multichannel_overlay`| 'cell' <br> 'bead' <br> 'cage' <br> 'nucleus' <br> 'cell-adhered' | 458, 416 | annotated as multi-channel overlaid images|
| 7 | `imr_90_cell_nucleus_cytoplasm_cage_set_6_458_416`|`240213_imr90_multichannel_overlay` | 'cell' <br> 'bead' <br> 'cage' <br> 'nucleus' <br> 'cell-adhered' | 458, 416 | annotated as multi-channel overlaid images|


| Model name | Training sets | LR scheduler | Test sets |
|:----------:|:-------------:|:------------:|:---------:|
|'cell_bead_cage_nucl_cyto_mix_crop_0p1_bbox_0p7_1_rs_0p25_blur_2_bs_8_epochs_2.pt' |1, 2, 3, 4, 5, 6 | 1, 2, 3, 4, 5   |'Step LR'|

* Mask RCNN Precision: 0.8997133278266363, Recall: 0.9471126796583295 at IoU 0.5 for labels: cell
* Mask RCNN F-1 Score: 0.922804744343666 at IoU 0.5 for labels: cell

* Mask RCNN Precision: 0.9779136165823963, Recall: 0.9642095342030854 at IoU 0.5 for labels: bead
* Mask RCNN F-1 Score: 0.9710132257621361 at IoU 0.5 for labels: bead

* Mask RCNN Precision: 0.9869942196531792, Recall: 0.9915795586527294 at IoU 0.5 for labels: cage
* Mask RCNN F-1 Score: 0.9892815758980302 at IoU 0.5 for labels: cage

* Mask RCNN Precision: 0.8980988593155893, Recall: 0.7915549597855228 at IoU 0.5 for labels: nucleus
* Mask RCNN F-1 Score: 0.8414677591734948 at IoU 0.5 for labels: nucleus

* Mask RCNN Precision: 0.47413793103448276, Recall: 0.3810623556581986 at IoU 0.5 for labels: cell-adhered
* Mask RCNN F-1 Score: 0.4225352112676057 at IoU 0.5 for labels: cell-adhered

| Model name | Training sets | LR scheduler | Test sets |
|:----------:|:-------------:|:------------:|:---------:|
|'cell_bead_cage_nucl_cyto_mix_crop_0p1_bbox_0p7_1_rs_0p25_blur_2_bs_8_epochs_1cl_lrs' |1, 2, 3, 4, 5, 6 | 1, 2, 3, 4, 5   |'1-Cycle LR'|


* Mask RCNN Precision: 0.9034226844392363, Recall: 0.9463966037542837 at IoU 0.5 for labels: cell
* Mask RCNN F-1 Score: 0.9244104716227018 at IoU 0.5 for labels: cell

* Mask RCNN Precision: 0.9774243072508998, Recall: 0.9632422243166824 at IoU 0.5 for labels: bead
* Mask RCNN F-1 Score: 0.9702814455784438 at IoU 0.5 for labels: bead

* Mask RCNN Precision: 0.9821736630247269, Recall: 0.991869918699187 at IoU 0.5 for labels: cage
* Mask RCNN F-1 Score: 0.9869979774631609 at IoU 0.5 for labels: cage

* Mask RCNN Precision: 0.9031273836765827, Recall: 0.7935656836461126 at IoU 0.5 for labels: nucleus
* Mask RCNN F-1 Score: 0.8448091330717089 at IoU 0.5 for labels: nucleus

* Mask RCNN Precision: 0.5004703668861712, Recall: 0.4095458044649731 at IoU 0.5 for labels: cell-adhered
* Mask RCNN F-1 Score: 0.4504657070279424 at IoU 0.5 for labels: cell-adhered

| Model name | Training sets | LR scheduler | Test sets |
|:----------:|:-------------:|:------------:|:---------:|
|'cell_bead_cage_nucl_cyto_mix_crop_0p1_bbox_0p7_1_rs_0p25_blur_2_bs_8_epochs_2.pt' |1, 2, 3, 4, 5, 6 | 1, 2, 3  |'Step LR'|

* Mask RCNN Precision: 0.0, Recall: 0.0 at IoU 0.5 for labels: cell

* Mask RCNN Precision: 0.0, Recall: 0.0 at IoU 0.5 for labels: bead

* Mask RCNN Precision: 0.8518518518518519, Recall: 0.92 at IoU 0.5 for labels: cage
* Mask RCNN F-1 Score: 0.8846153846153846 at IoU 0.5 for labels: cage

* Mask RCNN Precision: 0.8980988593155893, Recall: 0.7915549597855228 at IoU 0.5 for labels: nucleus
* Mask RCNN F-1 Score: 0.8414677591734948 at IoU 0.5 for labels: nucleus

* Mask RCNN Precision: 0.4755043227665706, Recall: 0.3810623556581986 at IoU 0.5 for labels: cell-adhered
* Mask RCNN F-1 Score: 0.42307692307692313 at IoU 0.5 for labels: cell-adhered


| Model name | Training sets | LR scheduler | Test sets |
|:----------:|:-------------:|:------------:|:---------:|
|'cell_bead_cage_nucl_cyto_mix_crop_0p1_bbox_0p7_1_rs_0p25_blur_2_bs_8_epochs_1cl_lrs.pt' |1, 2, 3, 4, 5, 6 | 1, 2, 3  |'1-Cycle LR'|

* Mask RCNN Precision: 0.0, Recall: 0.0 at IoU 0.5 for labels: cell

* Mask RCNN Precision: 0.0, Recall: 0.0 at IoU 0.5 for labels: bead

* Mask RCNN Precision: 0.8518518518518519, Recall: 0.92 at IoU 0.5 for labels: cage
* Mask RCNN F-1 Score: 0.8846153846153846 at IoU 0.5 for labels: cage

* Mask RCNN Precision: 0.9031273836765827, Recall: 0.7935656836461126 at IoU 0.5 for labels: nucleus
* Mask RCNN F-1 Score: 0.8448091330717089 at IoU 0.5 for labels: nucleus

* Mask RCNN Precision: 0.5009416195856874, Recall: 0.4095458044649731 at IoU 0.5 for labels: cell-adhered
* Mask RCNN F-1 Score: 0.4506565014824227 at IoU 0.5 for labels: cell-adhered


Combined model for cells, beads, cages, nuclei and cell-adhered
Overlaid set (IMR-90 nuclei set 4, 5) test

Mask RCNN Precision: 0.6796875, Recall: 0.8743718592964824 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.7648351648351649 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.0, Recall: 0.0 at IoU 0.5 for labels: bead

Mask RCNN Precision: 0.9541062801932367, Recall: 0.9875 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9705159705159706 at IoU 0.5 for labels: cage

Mask RCNN Precision: 0.8928571428571429, Recall: 0.8566978193146417 at IoU 0.5 for labels: nucleus
Mask RCNN F-1 Score: 0.8744038155802861 at IoU 0.5 for labels: nucleus

Mask RCNN Precision: 0.7420289855072464, Recall: 0.8737201365187713 at IoU 0.5 for labels: cell-adhered
Mask RCNN F-1 Score: 0.8025078369905956 at IoU 0.5 for labels: cell-adhered

New cropping 1-Cycle LRS
Mask RCNN Precision: 0.7049180327868853, Recall: 0.864321608040201 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.7765237020316028 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.0, Recall: 0.0 at IoU 0.5 for labels: bead

Mask RCNN Precision: 0.9541062801932367, Recall: 0.9875 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9705159705159706 at IoU 0.5 for labels: cage

Mask RCNN Precision: 0.846875, Recall: 0.8442367601246106 at IoU 0.5 for labels: nucleus
Mask RCNN F-1 Score: 0.8455538221528862 at IoU 0.5 for labels: nucleus

Mask RCNN Precision: 0.8118811881188119, Recall: 0.8395904436860068 at IoU 0.5 for labels: cell-adhered
Mask RCNN F-1 Score: 0.825503355704698 at IoU 0.5 for labels: cell-adhered

New cropping Step LRS
Mask RCNN Precision: 0.7160493827160493, Recall: 0.8743718592964824 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.7873303167420813 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.0, Recall: 0.0 at IoU 0.5 for labels: bead

Mask RCNN Precision: 0.9588377723970944, Recall: 0.99 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.974169741697417 at IoU 0.5 for labels: cage

Mask RCNN Precision: 0.8679245283018868, Recall: 0.8598130841121495 at IoU 0.5 for labels: nucleus
Mask RCNN F-1 Score: 0.863849765258216 at IoU 0.5 for labels: nucleus

Mask RCNN Precision: 0.8066666666666666, Recall: 0.825938566552901 at IoU 0.5 for labels: cell-adhered
Mask RCNN F-1 Score: 0.8161888701517707 at IoU 0.5 for labels: cell-adhered


Combined model for cells, beads, cages, nuclei and cell-adhered
Combined set (caging set + analysis set 1 + analysis set 2 + IMR-90 set 1, 2, 3, 4, 5 with 4, 5 
being the overlaid images) test

Mask RCNN Precision: 0.8839408773347275, Recall: 0.9417215189873418 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9119168444019514 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9815585349565932, Recall: 0.9478644774046332 at IoU 0.5 for labels: bead
Mask RCNN F-1 Score: 0.9644173017715642 at IoU 0.5 for labels: bead

Mask RCNN Precision: 0.9879454926624738, Recall: 0.9807492195629552 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9843342036553524 at IoU 0.5 for labels: cage

Mask RCNN Precision: 0.9136735979836169, Recall: 0.7997793712079426 at IoU 0.5 for labels: nucleus
Mask RCNN F-1 Score: 0.8529411764705881 at IoU 0.5 for labels: nucleus

Mask RCNN Precision: 0.47537091988130564, Recall: 0.503140703517588 at IoU 0.5 for labels: cell-adhered
Mask RCNN F-1 Score: 0.4888617638083613 at IoU 0.5 for labels: cell-adhered

New cropping 1-Cycle LRS
Mask RCNN Precision: 0.9010856453558505, Recall: 0.9455696202531646 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9227918468190243 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9773997080585897, Recall: 0.9632422243166824 at IoU 0.5 for labels: bead
Mask RCNN F-1 Score: 0.9702693249387898 at IoU 0.5 for labels: bead

Mask RCNN Precision: 0.9791880781089414, Recall: 0.9914151925078044 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9852637021716649 at IoU 0.5 for labels: cage

Mask RCNN Precision: 0.8920907418761496, Recall: 0.8025372311086597 at IoU 0.5 for labels: nucleus
Mask RCNN F-1 Score: 0.8449477351916376 at IoU 0.5 for labels: nucleus

Mask RCNN Precision: 0.5695461200585652, Recall: 0.4886934673366834 at IoU 0.5 for labels: cell-adhered
Mask RCNN F-1 Score: 0.5260311020960108 at IoU 0.5 for labels: cell-adhered

New cropping Step LRS
Mask RCNN Precision: 0.8975701114099116, Recall: 0.946379746835443 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9213289298565583 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9778644195698655, Recall: 0.9642095342030854 at IoU 0.5 for labels: bead
Mask RCNN F-1 Score: 0.9709889725625366 at IoU 0.5 for labels: bead

Mask RCNN Precision: 0.9839917376710561, Recall: 0.9914151925078044 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9876895166515485 at IoU 0.5 for labels: cage

Mask RCNN Precision: 0.8922229026331905, Recall: 0.8036403750689465 at IoU 0.5 for labels: nucleus
Mask RCNN F-1 Score: 0.8456181079512478 at IoU 0.5 for labels: nucleus

Mask RCNN Precision: 0.5483630952380952, Recall: 0.4629396984924623 at IoU 0.5 for labels: cell-adhered
Mask RCNN F-1 Score: 0.5020435967302452 at IoU 0.5 for labels: cell-adhered

Combined model for cells, beads, cages, nuclei, cell-adhered and hela cells (as a separate class)
Overlaid set (IMR-90 nuclei set 4, 5) test

New cropping 1-Cycle LRS
Mask RCNN Precision: 0.678030303030303, Recall: 0.8994974874371859 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.773218142548596 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.0, Recall: 0.0 at IoU 0.5 for labels: bead

Mask RCNN Precision: 0.948780487804878, Recall: 0.9725 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9604938271604938 at IoU 0.5 for labels: cage

Mask RCNN Precision: 0.865814696485623, Recall: 0.8442367601246106 at IoU 0.5 for labels: nucleus
Mask RCNN F-1 Score: 0.8548895899053628 at IoU 0.5 for labels: nucleus

Mask RCNN Precision: 0.7745098039215687, Recall: 0.8088737201365188 at IoU 0.5 for labels: cell-adhered
Mask RCNN F-1 Score: 0.7913188647746243 at IoU 0.5 for labels: cell-adhered


Combined model for cells, beads, cages, nuclei, cell-adhered and hela cells (as a separate class)
Combined set (caging set + analysis set 1 + analysis set 2 + IMR-90 set 1, 2, 3, 4, 5 with 4, 5 
being the overlaid images) test

New cropping 1-Cycle LRS
Mask RCNN Precision: 0.8996539792387543, Recall: 0.9478481012658228 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9231224419350066 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.97728871470204, Recall: 0.9648296046430874 at IoU 0.5 for labels: bead
Mask RCNN F-1 Score: 0.9710191957265171 at IoU 0.5 for labels: bead

Mask RCNN Precision: 0.9837167226673559, Recall: 0.9901144640998959 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9869052249448982 at IoU 0.5 for labels: cage

Mask RCNN Precision: 0.9001233045622689, Recall: 0.8052950910093767 at IoU 0.5 for labels: nucleus
Mask RCNN F-1 Score: 0.8500727802037846 at IoU 0.5 for labels: nucleus

Mask RCNN Precision: 0.5508537490720119, Recall: 0.46608040201005024 at IoU 0.5 for labels: cell-adhered
Mask RCNN F-1 Score: 0.5049336509016672 at IoU 0.5 for labels: cell-adhered

Combined model for cells, beads, cages, nuclei, cell-adhered and hela cells (as a separate class)
hela cells test dataset

New cropping 1-Cycle LRS
Mask RCNN Precision: 0.99185667752443, Recall: 0.9967266775777414 at IoU 0.5 for labels: bead
Mask RCNN F-1 Score: 0.9942857142857142 at IoU 0.5 for labels: bead

Mask RCNN Precision: 1.0, Recall: 1.0 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 1.0 at IoU 0.5 for labels: cage

Mask RCNN Precision: 0.8313500205170291, Recall: 0.9173647271904007 at IoU 0.5 for labels: hela-cell
Mask RCNN F-1 Score: 0.8722419545797008 at IoU 0.5 for labels: hela-cell


In [ ]:
img = cv2.imread('2_2_4_627_1_054183_015230_-15234_White.caging.png', cv2.IMREAD_UNCHANGED)

In [ ]:
import cv2
import numpy as np
from PIL import Image
import sys
sys.path.append("../utils")
from cv_utils import show_detections
from yolo_model import run_yolo_v5

In [ ]:
path: str = '/home/cellareye/Cellanome/Data/MC38/20240522_mc38-adhered-bfcaging_10x_uncaged/'

In [ ]:
imgs = os.listdir(path)
imgs = [img for img in imgs if '_White' in img]

In [ ]:
idx = 4
img = cv2.imread(os.path.join(path, imgs[idx]), cv2.IMREAD_UNCHANGED)
img = (img.astype(float) / (2**12 - 1) * 255).astype(np.uint8)

In [ ]:
Image.fromarray(img)

In [ ]:
boxes, labels, scores, run_time = run_yolo_v5(input_image=img, normalize_image=True, is_4x=False)

In [ ]:
out = show_detections(input_image=img, predictions={'boxes': boxes, 'labels': labels, 'scores': scores}, label_map={0:'cell', 1:'bead', 2:'somae'}) 

In [ ]:
Image.fromarray(out)

In [ ]:
img = cv2.imread('2_2_4_627_1_054183_015230_-15234_White.caging.png', cv2.IMREAD_UNCHANGED)
img = (img.astype(float) / (2**12 - 1) * 255).astype(np.uint8)

In [ ]:
import time
start = time.time()
